# 06 — CAC × LTV por canal

La página de cierre. Las cinco anteriores construyeron las dos mitades de una misma pregunta:

- la **página 5** midió cuánto vale un suscriptor según el canal por el que entró, proyectando la
  curva de retención de cada canal;
- la **página 6** midió cuánto cuesta captarlo, repartiendo el gasto entre canales y contando lo que
  no se puede repartir.

Aquí se dividen. Y la tesis del proyecto es que esa división **cambia el orden de los canales**
respecto a mirar sólo el coste: un canal que capta barato y retiene mal es caro, y sólo se ve
cruzando las dos cosas.

Esta página no vuelve al warehouse: consume los dos JSON que dejaron las anteriores, que es
exactamente la dependencia que el proyecto quiere tener entre notebooks —de datos, no de orden de
ejecución—.

In [1]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data" / "warehouse.duckdb").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUTS = PROJECT_ROOT / "analysis" / "outputs"
OUTPUT_PATH = OUTPUTS / "cac_ltv.json"

# --- Parámetros del análisis ---
HEALTHY_RATIO = 3.0        # regla del pulgar habitual en suscripción
PAYBACK_TARGET_MONTHS = 12
MARGIN_SCENARIOS = (0.10, 0.30, 0.50, 0.70)

C_BLUE, C_ORANGE, C_AQUA, C_YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
C_VIOLET, C_RED = "#4a3aa7", "#e34948"
C_GRID, C_INK, C_MUTED = "#e6e6e3", "#0b0b0b", "#52514e"
CHANNEL_COLOR = {"paid_social": C_BLUE, "organic": C_AQUA, "influencer_code": C_ORANGE,
                 "referral": C_VIOLET, "podcast_ads": C_YELLOW, "direct_unknown": C_MUTED}
CHANNEL_LABEL = {"paid_social": "Paid social", "organic": "Orgánico",
                 "influencer_code": "Código influencer", "referral": "Referido",
                 "podcast_ads": "Podcast", "direct_unknown": "Directo / sin resolver"}

PLOT_LAYOUT = dict(
    template="plotly_white", height=420,
    margin=dict(l=70, r=30, t=60, b=50),
    font=dict(color=C_INK, size=12),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    xaxis=dict(gridcolor=C_GRID), yaxis=dict(gridcolor=C_GRID),
    hovermode="closest",
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

attribution = json.loads((OUTPUTS / "attribution.json").read_text(encoding="utf-8"))
cohorts = json.loads((OUTPUTS / "cohorts_rfm.json").read_text(encoding="utf-8"))
print("insumos:")
for name, payload in (("attribution.json", attribution), ("cohorts_rfm.json", cohorts)):
    print(f"  {name:20s} generado {payload['meta']['generated_at']}")
print()
print("ventana de atribución:", attribution["meta"]["censoring_cutoff"])
print("horizonte del LTV:", cohorts["meta"]["ltv_horizon_months"], "meses")

insumos:
  attribution.json     generado 2026-09-19T23:43:38+00:00
  cohorts_rfm.json     generado 2026-09-19T23:05:01+00:00

ventana de atribución: 2026-05-31
horizonte del LTV: 36 meses


## 1. Qué se cruza exactamente

Las dos mitades no son números sueltos: cada una arrastra una decisión metodológica que la página
anterior dejó documentada, y las dos decisiones cambian el resultado.

| | Se usa | La alternativa era | Por qué se descartó |
|---|---|---|---|
| **CAC** | Cargado con el gasto huérfano | Sólo el gasto asignable | El 24,5% del gasto no resuelve a ningún cliente; ignorarlo abarata a unos canales más que a otros. |
| **LTV** | Proyectado con la curva del canal | Observado sobre suscripciones maduras | El observado sólo promedia a los supervivientes, así que premia justo a los canales que peor retienen. |

Las dos alternativas se calculan igualmente, porque la sección 4 mide **cuánto se mueve la decisión**
según cuál se elija. Eso es lo que convierte el cruce en un análisis y no en una tabla.

In [2]:
cac_rows = pd.DataFrame(attribution["cac"]["by_channel"]).set_index("channel")
ltv_rows = pd.DataFrame(cohorts["ltv_by_channel"]["channels"]).set_index("canal")

cross = pd.DataFrame({
    "cac_reparto": cac_rows["markov"],
    "cac_cargado": cac_rows["cac_cargado_markov"],
    "conversiones": cac_rows["conversions_markov"],
    "coste_asignable": cac_rows["attributable_cost_eur"],
    "coste_huerfano": cac_rows["orphan_cost_eur"],
    "ltv_proyectado": ltv_rows["ltv_proyectado"],
    "ltv_observado_maduras": ltv_rows["ltv_observado_maduras"],
    "arpu_mes": ltv_rows["arpu_mes"],
    "meses_esperados": ltv_rows["meses_esperados"],
    "retencion_12m": ltv_rows["retencion_12m"],
    "suscripciones": ltv_rows["suscripciones"],
})
cross.index.name = "canal"

# Los canales sin coste de medios no admiten ratio: se tratan aparte en la sección 6.
priced = cross[cross.cac_cargado.fillna(0) > 0].copy()
unpriced = cross[~cross.index.isin(priced.index)].copy()

priced["ratio"] = priced.ltv_proyectado / priced.cac_cargado
priced["payback_meses"] = priced.cac_cargado / priced.arpu_mes
priced["margen_beneficio_eur"] = priced.ltv_proyectado - priced.cac_cargado
priced = priced.sort_values("ratio", ascending=False)

print("Canales con coste de medios:")
print(priced[["cac_cargado", "ltv_proyectado", "ratio", "payback_meses", "retencion_12m",
              "conversiones"]].round(2).to_string())
print()
print("Canales sin coste de medios en el dataset:")
print(unpriced[["ltv_proyectado", "retencion_12m", "suscripciones"]].round(2).to_string())

Canales con coste de medios:
                 cac_cargado  ltv_proyectado  ratio  payback_meses  retencion_12m  conversiones
canal                                                                                          
paid_social            14.44          539.07  37.33           0.57          64.57       1143.55
podcast_ads            13.52          454.96  33.65           0.52          54.40        679.54
influencer_code        19.09          411.42  21.55           0.76          52.47        861.03
referral               40.57          480.04  11.83           1.64          56.84        556.56

Canales sin coste de medios en el dataset:
                ltv_proyectado  retencion_12m  suscripciones
canal                                                       
direct_unknown          483.79          60.42           1401
organic                 443.00          58.94            914


## 2. El cruce

El gráfico clásico: coste en un eje, valor en el otro, y las diagonales de ratio constante. Un canal
está tanto mejor cuanto más arriba y más a la izquierda.

In [3]:
fig = go.Figure()
x_max = priced.cac_cargado.max() * 1.25
for ratio, dash in ((HEALTHY_RATIO, "solid"), (10, "dot"), (30, "dot")):
    fig.add_trace(go.Scatter(x=[0, x_max], y=[0, x_max * ratio], mode="lines",
                             line=dict(color=C_GRID, width=1.5, dash=dash),
                             name=f"ratio {ratio:g}x", hoverinfo="skip"))
for canal, row in priced.iterrows():
    fig.add_trace(go.Scatter(
        x=[row.cac_cargado], y=[row.ltv_proyectado], mode="markers+text",
        marker=dict(size=np.sqrt(row.conversiones) * 1.6, color=CHANNEL_COLOR[canal],
                    line=dict(color="white", width=1.5)),
        text=[CHANNEL_LABEL[canal]], textposition="top center",
        name=CHANNEL_LABEL[canal], showlegend=False,
        hovertemplate=(f"<b>{CHANNEL_LABEL[canal]}</b><br>CAC {row.cac_cargado:.2f} €<br>"
                       f"LTV {row.ltv_proyectado:.0f} €<br>ratio {row.ratio:.1f}x<br>"
                       f"{row.conversiones:.0f} conversiones<extra></extra>")))
fig.update_layout(**{**PLOT_LAYOUT, "height": 470},
                  title="CAC × LTV por canal (tamaño = conversiones atribuidas)",
                  xaxis_title="CAC cargado (€ por suscriptor)",
                  yaxis_title="LTV proyectado a 36 meses (€)")
fig.update_xaxes(range=[0, x_max])
fig.update_yaxes(range=[0, priced.ltv_proyectado.max() * 1.18])
fig.show()

cheapest = priced.cac_cargado.idxmin()
best = priced.ratio.idxmax()
print(f"canal más barato de captar : {CHANNEL_LABEL[cheapest]} ({priced.cac_cargado.min():.2f} €)")
print(f"canal con mejor ratio      : {CHANNEL_LABEL[best]} ({priced.ratio.max():.1f}x)")
print(f"¿coinciden? {'sí' if cheapest == best else 'NO'}")

canal más barato de captar : Podcast (13.52 €)
canal con mejor ratio      : Paid social (37.3x)
¿coinciden? NO


**Y no coinciden, que es exactamente la tesis del proyecto.**

El **podcast es el canal más barato** de captar: 13,52 € por suscriptor, un 6% menos que paid social.
Cualquier cuadro de mando que ordene canales por CAC lo pondría el primero y recomendaría moverle
presupuesto.

Pero **paid social vale más**: 539 € de LTV frente a 455 €, y por tanto un ratio de **37,3x frente a
33,7x**. El orden se invierte al cruzar, y se invierte por una razón que la página 5 ya había medido
—la retención— no por un artefacto del cálculo.

En el otro extremo, `referral` es el caso claro: cuesta **tres veces más** que cualquier otro canal
(40,57 €) y su LTV es intermedio, así que su ratio (11,8x) es un tercio del de paid social. Es el
único canal donde el coste de captación empieza a ser un problema de verdad.

## 3. Por qué cambia el orden: el LTV es retención, no tarifa

Conviene abrir el LTV para ver de dónde sale la diferencia entre canales, porque determina qué
palanca hay que tocar. `LTV = ARPU mensual × meses de vida esperados`, y los dos factores no
contribuyen igual.

In [4]:
decomposition = priced[["arpu_mes", "meses_esperados", "ltv_proyectado", "retencion_12m"]].copy()
spread = pd.Series({
    "ARPU mensual": priced.arpu_mes.max() / priced.arpu_mes.min() - 1,
    "meses esperados": priced.meses_esperados.max() / priced.meses_esperados.min() - 1,
    "LTV": priced.ltv_proyectado.max() / priced.ltv_proyectado.min() - 1,
})
print(decomposition.round(2).to_string())
print()
print("Dispersión entre el mejor y el peor canal:")
print((spread * 100).round(1).to_string())

# ¿Qué fracción de la diferencia de LTV explica cada factor? Descomposición logarítmica.
best_c, worst_c = priced.ltv_proyectado.idxmax(), priced.ltv_proyectado.idxmin()
log_gap = np.log(priced.ltv_proyectado[best_c] / priced.ltv_proyectado[worst_c])
log_arpu = np.log(priced.arpu_mes[best_c] / priced.arpu_mes[worst_c])
log_months = np.log(priced.meses_esperados[best_c] / priced.meses_esperados[worst_c])
print()
print(f"Diferencia de LTV entre {CHANNEL_LABEL[best_c]} y {CHANNEL_LABEL[worst_c]}:")
print(f"  por ARPU            : {log_arpu / log_gap:.1%}")
print(f"  por meses de vida   : {log_months / log_gap:.1%}")

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=("LTV = ARPU × meses de vida esperados",
                                    "Retención a 12 meses y ratio LTV:CAC"))
order = priced.ltv_proyectado.sort_values().index
fig.add_trace(go.Bar(y=[CHANNEL_LABEL[c] for c in order], x=priced.arpu_mes[order] * priced.meses_esperados[order],
                     orientation="h", marker_color=[CHANNEL_COLOR[c] for c in order],
                     text=[f"{priced.ltv_proyectado[c]:.0f} €" for c in order],
                     textposition="outside", showlegend=False,
                     hovertemplate="%{y}: %{x:.0f} €<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=priced.retencion_12m, y=priced.ratio, mode="markers+text",
                         marker=dict(size=16, color=[CHANNEL_COLOR[c] for c in priced.index]),
                         text=[CHANNEL_LABEL[c] for c in priced.index], textposition="top center",
                         showlegend=False,
                         hovertemplate="%{text}<br>retención 12m %{x:.1f}%<br>ratio %{y:.1f}x<extra></extra>"),
              row=1, col=2)
fig.update_xaxes(title_text="€", row=1, col=1)
fig.update_xaxes(title_text="% retenido a 12 meses", row=1, col=2)
fig.update_yaxes(title_text="ratio LTV:CAC", row=1, col=2)
fig.update_layout(**{**PLOT_LAYOUT, "height": 420},
                  title="La diferencia de valor entre canales es casi toda retención")
fig.show()

                 arpu_mes  meses_esperados  ltv_proyectado  retencion_12m
canal                                                                    
paid_social         25.53            21.11          539.07          64.57
podcast_ads         25.98            17.51          454.96          54.40
influencer_code     25.03            16.43          411.42          52.47
referral            24.79            19.37          480.04          56.84

Dispersión entre el mejor y el peor canal:
ARPU mensual        4.8
meses esperados    28.5
LTV                31.0

Diferencia de LTV entre Paid social y Código influencer:
  por ARPU            : 7.3%
  por meses de vida   : 92.7%


**El ARPU es prácticamente el mismo en todos los canales** —de 24,79 € a 25,98 €, un 4,8% de
diferencia— mientras que los **meses de vida esperados van de 16,4 a 21,1, un 28%**. Descomponiendo
la diferencia de LTV entre el mejor y el peor canal, el **93% viene de la duración** y sólo el 7% de
la tarifa.

Tiene una consecuencia práctica directa: **el canal no cambia lo que el cliente paga, cambia cuánto
tiempo se queda.** Así que la palanca para mejorar el LTV de un canal no es el pricing —que es común
a todos— sino la calidad del cliente que trae. Y eso se decide en la captación, no en el producto.

El panel de la derecha lo enseña sin intermediarios: el ratio LTV:CAC ordena los canales casi igual
que la retención a 12 meses. Paid social retiene un 64,6% y encabeza; el código de influencer retiene
un 52,5% y es tercero pese a costar un 32% menos que referral.

## 4. ¿Aguanta la decisión las dos elecciones metodológicas?

El ranking de la sección 2 sale de haber elegido **CAC cargado** y **LTV proyectado**. Las dos
elecciones están justificadas en las páginas anteriores, pero un ranking que sólo se sostiene con la
combinación correcta es un ranking frágil. Conviene calcular las cuatro.

In [5]:
combos = {
    "cargado × proyectado": priced.ltv_proyectado / priced.cac_cargado,
    "reparto × proyectado": priced.ltv_proyectado / priced.cac_reparto,
    "cargado × obs. maduras": priced.ltv_observado_maduras / priced.cac_cargado,
    "reparto × obs. maduras": priced.ltv_observado_maduras / priced.cac_reparto,
}
ratios = pd.DataFrame(combos)
ranks = ratios.rank(ascending=False).astype(int)
print("Ratio LTV:CAC según las dos elecciones:")
print(ratios.round(1).to_string())
print()
print("Puesto en el ranking:")
print(ranks.to_string())
print()
movers = ranks[ranks.nunique(axis=1) > 1]
print(f"canales que cambian de puesto: {len(movers)} de {len(ranks)}")
for canal in movers.index:
    print(f"  {CHANNEL_LABEL[canal]}: puestos {sorted(set(ranks.loc[canal]))}")
print()
print(f"el ratio de un mismo canal varía hasta un "
      f"{(ratios.max(axis=1) / ratios.min(axis=1) - 1).max():.0%} según la combinación")

fig = go.Figure()
for canal in ratios.index:
    fig.add_trace(go.Scatter(x=list(ratios.columns), y=ratios.loc[canal], mode="lines+markers",
                             name=CHANNEL_LABEL[canal],
                             line=dict(color=CHANNEL_COLOR[canal], width=2.5),
                             marker=dict(size=9)))
fig.add_hline(y=HEALTHY_RATIO, line=dict(color=C_RED, width=1.5, dash="dash"),
              annotation_text=f"regla del pulgar {HEALTHY_RATIO:g}x", annotation_position="bottom left")
fig.update_layout(**{**PLOT_LAYOUT, "height": 420, "hovermode": "x unified"},
                  title="El mismo cruce, con las cuatro combinaciones metodológicas",
                  yaxis_title="ratio LTV:CAC")
fig.show()

Ratio LTV:CAC según las dos elecciones:
                 cargado × proyectado  reparto × proyectado  cargado × obs. maduras  reparto × obs. maduras
canal                                                                                                      
paid_social                      37.3                  53.9                    38.1                    55.1
podcast_ads                      33.7                  44.3                    42.4                    55.8
influencer_code                  21.5                  27.3                    28.1                    35.6
referral                         11.8                  15.3                    13.6                    17.6

Puesto en el ranking:
                 cargado × proyectado  reparto × proyectado  cargado × obs. maduras  reparto × obs. maduras
canal                                                                                                      
paid_social                         1                     1              

**La conclusión de negocio aguanta; el podio, no del todo.**

`referral` es el peor canal en las cuatro combinaciones y el código de influencer es tercero en las
cuatro: eso no depende de ninguna elección. Pero **paid social y podcast se intercambian el primer
puesto** según qué LTV se use: con el proyectado gana paid social (37,3x contra 33,7x) y con el
observado sobre maduras gana el podcast (42,4x contra 38,1x).

No es casualidad ni ruido: es **exactamente el sesgo que la página 5 documentó**. El LTV observado
sobre suscripciones maduras sólo promedia a las que llegaron a los 12 meses, así que premia al canal
que peor retiene —el podcast retiene un 54,4% y paid social un 64,6%—. Elegir el LTV equivocado no
"mete ruido": **le da la victoria al canal que la pierde**.

Y el orden de magnitud del error es grande: el ratio de un mismo canal varía hasta un **66%** según
la combinación. Cualquiera de las cuatro tablas tiene el mismo aspecto de rigor en una diapositiva.

## 5. El ratio es demasiado bueno, y eso también es un hallazgo

La regla del pulgar en suscripción es que un ratio LTV:CAC por debajo de 3x no se sostiene y por
encima de 5x sugiere que se está invirtiendo de menos. Aquí el peor canal está en **11,8x** y el
mejor en **37,3x**.

Publicar eso como "el marketing va estupendamente" sería el error de esta página. Un ratio así
normalmente significa que falta un coste en la cuenta, así que antes de celebrarlo hay que ir a
buscarlo.

In [6]:
marketing_spend = attribution["meta"]["total_cost_all_history_eur"]
total_revenue = (1308457 + 1758474 + 840859)   # suscripción + tienda + máquina, páginas 2 y 3
print(f"gasto de marketing (histórico completo): {marketing_spend:,.0f} €")
print(f"ingreso total del histórico:             {total_revenue:,.0f} €")
print(f"intensidad de marketing:                 {marketing_spend / total_revenue:.2%} del ingreso")
print("  (en suscripción/DTC lo habitual es un 10-30%)")
print()
print("El dataset no tiene coste de producto: `int_product_sku_continuity` sólo lleva precio de")
print("lista. El LTV de esta página es, por tanto, **ingreso**, no margen de contribución.")
print()

margin_table = pd.DataFrame(
    {f"margen {m:.0%}": priced.ltv_proyectado * m / priced.cac_cargado for m in MARGIN_SCENARIOS})
print("Ratio LTV:CAC si el LTV fuera margen y no ingreso:")
print(margin_table.round(1).to_string())
print()
breakeven = (HEALTHY_RATIO * priced.cac_cargado / priced.ltv_proyectado * 100).sort_values()
print(f"Margen bruto al que cada canal caería al umbral de {HEALTHY_RATIO:g}x:")
print(breakeven.round(1).to_string())

fig = go.Figure()
for canal in priced.index:
    fig.add_trace(go.Scatter(x=[m * 100 for m in MARGIN_SCENARIOS],
                             y=[priced.ltv_proyectado[canal] * m / priced.cac_cargado[canal]
                                for m in MARGIN_SCENARIOS],
                             mode="lines+markers", name=CHANNEL_LABEL[canal],
                             line=dict(color=CHANNEL_COLOR[canal], width=2.5)))
fig.add_hline(y=HEALTHY_RATIO, line=dict(color=C_RED, width=1.5, dash="dash"),
              annotation_text=f"umbral {HEALTHY_RATIO:g}x")
fig.update_layout(**{**PLOT_LAYOUT, "height": 400, "hovermode": "x unified"},
                  title="Ratio LTV:CAC según el margen bruto que se suponga",
                  xaxis_title="margen bruto supuesto (%)", yaxis_title="ratio LTV:CAC")
fig.show()

gasto de marketing (histórico completo): 70,767 €
ingreso total del histórico:             3,907,790 €
intensidad de marketing:                 1.81% del ingreso
  (en suscripción/DTC lo habitual es un 10-30%)

El dataset no tiene coste de producto: `int_product_sku_continuity` sólo lleva precio de
lista. El LTV de esta página es, por tanto, **ingreso**, no margen de contribución.

Ratio LTV:CAC si el LTV fuera margen y no ingreso:
                 margen 10%  margen 30%  margen 50%  margen 70%
canal                                                          
paid_social             3.7        11.2        18.7        26.1
podcast_ads             3.4        10.1        16.8        23.6
influencer_code         2.2         6.5        10.8        15.1
referral                1.2         3.5         5.9         8.3

Margen bruto al que cada canal caería al umbral de 3x:
canal
paid_social         8.0
podcast_ads         8.9
influencer_code    13.9
referral           25.4


**Los dos costes que faltan explican el ratio, y ninguno invalida la comparación entre canales.**

El primero es el **margen**: sin coste de producto en el dataset, el LTV de esta página es ingreso
bruto. Si el margen de contribución fuera del 30% —razonable para cápsulas con logística— los ratios
caerían a un rango de **3,5x a 11,2x**, que ya está en territorio creíble.

El segundo es la **intensidad de marketing**: 70.767 € de gasto ponderado sobre 3,9 M€ de ingreso es
un **1,81%**, cuando en suscripción lo normal es un 10-30%. Este negocio, tal y como está simulado,
capta clientes casi gratis.

La tabla de sensibilidad da la respuesta útil: **paid social seguiría por encima del umbral de 3x
aunque el margen bruto fuera del 8%**, y el podcast del 8,9%. `referral` necesita un **25,4%** de
margen para justificarse, que es el único caso donde un supuesto plausible cambia la decisión.

Y la lectura de negocio, que es incómoda pero es la que dicen los números: **con estos ratios el
cuello de botella no es el coste de captación, es el volumen.** La recomendación no es optimizar el
CAC, es gastar más en los canales que aguantan el escrutinio —siempre que el CAC no se dispare al
escalar, que es un supuesto que este dataset no permite comprobar—.

## 6. Los canales sin precio

Dos canales no tienen coste de medios y por tanto no tienen ratio. Meterlos en la tabla con un CAC de
0 € los pondría primeros con ratio infinito, que es la conclusión más tonta que puede sacar esta
página.

In [7]:
print("Canales sin coste de medios:")
for canal, row in unpriced.iterrows():
    print(f"  {CHANNEL_LABEL[canal]:24s} LTV {row.ltv_proyectado:6.0f} € · "
          f"retención 12m {row.retencion_12m:.1f}% · {int(row.suscripciones):,} suscripciones")
print()
attributed = int(priced.suscripciones.sum())
unattributed = int(unpriced.suscripciones.sum())
print(f"suscripciones con canal de pago: {attributed:,} · sin coste asociado: {unattributed:,} "
      f"({unattributed / (attributed + unattributed):.0%} del total)")
print()
print("valor anual en juego si el orgánico mantuviera su retención:")
print(f"  {unpriced.loc['organic', 'ltv_proyectado'] * unpriced.loc['organic', 'suscripciones']:,.0f} €"
      if "organic" in unpriced.index else "  n/a")

Canales sin coste de medios:
  Directo / sin resolver   LTV    484 € · retención 12m 60.4% · 1,401 suscripciones
  Orgánico                 LTV    443 € · retención 12m 58.9% · 914 suscripciones

suscripciones con canal de pago: 2,468 · sin coste asociado: 2,315 (48% del total)

valor anual en juego si el orgánico mantuviera su retención:
  404,904 €


**`organic` (443 € de LTV, 58,9% de retención a 12 meses) es un canal de calidad media-alta que no
aparece en ninguna tabla de eficiencia** porque su coste —contenido, SEO, marca— no está en el
dataset. Con 914 suscripciones captadas, no es marginal: es el segundo canal por volumen.

Lo honesto es sacarlo de la comparación de ratios y decir qué haría falta para incluirlo: imputarle
el coste del equipo y del contenido, que es un ejercicio de contabilidad analítica, no de atribución.

**`direct_unknown` es otra cosa**: no es un canal sino el 29% de las altas cuyo origen no se pudo
resolver. Su LTV (484 €) es informativo —dice que no son clientes peores que la media— pero no se le
puede asignar coste ni optimizar. Es el recordatorio de que **casi un tercio de la captación de este
negocio está fuera del alcance de cualquier modelo de atribución**, por bueno que sea.

## 7. Dónde va el siguiente euro

In [8]:
recommendation = priced[["cac_cargado", "ltv_proyectado", "ratio", "payback_meses",
                         "retencion_12m", "conversiones"]].copy()
recommendation["margen_por_alta"] = priced.margen_beneficio_eur
recommendation["veredicto"] = np.where(
    recommendation.ratio >= 30, "escalar",
    np.where(recommendation.ratio >= 15, "mantener y vigilar la retención", "revisar el coste"))
print(recommendation.round(2).to_string())
print()
print(f"payback más lento: {recommendation.payback_meses.max():.1f} meses "
      f"({CHANNEL_LABEL[recommendation.payback_meses.idxmax()]}), muy por debajo del objetivo "
      f"de {PAYBACK_TARGET_MONTHS} meses")
print()
blended_cac = attribution["cac"]["blended_loaded_eur"]
blended_ltv = float((priced.ltv_proyectado * priced.conversiones).sum() / priced.conversiones.sum())
print(f"CAC medio cargado {blended_cac:.2f} € · LTV medio ponderado {blended_ltv:.0f} € · "
      f"ratio {blended_ltv / blended_cac:.1f}x")

                 cac_cargado  ltv_proyectado  ratio  payback_meses  retencion_12m  conversiones  margen_por_alta                        veredicto
canal                                                                                                                                            
paid_social            14.44          539.07  37.33           0.57          64.57       1143.55           524.63                          escalar
podcast_ads            13.52          454.96  33.65           0.52          54.40        679.54           441.44                          escalar
influencer_code        19.09          411.42  21.55           0.76          52.47        861.03           392.33  mantener y vigilar la retención
referral               40.57          480.04  11.83           1.64          56.84        556.56           439.47                 revisar el coste

payback más lento: 1.6 meses (Referido), muy por debajo del objetivo de 12 meses

CAC medio cargado 14.53 € · LTV medio pon

Con las salvedades de la sección 5 —margen ausente e intensidad de marketing anormalmente baja— la
lectura por canal es:

- **Paid social: escalar.** Mejor ratio (37,3x), mejor retención (64,6%), mayor volumen ya captado y
  además su CAC está **sobrestimado**, porque la ventana de pérdida de trazabilidad de 2025 le quitó
  conversiones que sí consiguió. Es el canal con más margen para absorber inversión.
- **Podcast: escalar con vigilancia.** Es el más barato y su ratio es el segundo, pero su ventaja
  depende de qué LTV se use y su retención es 10 puntos peor. Merece más presupuesto y una medición
  de retención a 12 meses antes de doblarlo.
- **Código de influencer: mantener.** Ratio de 21,5x, correcto, pero es el canal que **peor retiene**
  del catálogo (52,5%). Antes de escalarlo conviene entender si el problema es el tipo de creador o
  el tipo de promoción, porque es una diferencia de 5 meses de vida frente a paid social.
- **Referido: revisar el coste.** 40,57 € por alta, tres veces el resto, y el único canal que caería
  por debajo del umbral con un margen del 25%. No es que no funcione —retiene bien— es que el
  incentivo está caro.

El agregado: **14,53 € de CAC cargado contra 483 € de LTV ponderado, un ratio de 33x**. Y el payback
más lento del catálogo es de **1,6 meses**, frente a un objetivo habitual de 12. Si hay una conclusión
de negocio en todo el proyecto, es que este negocio no tiene un problema de coste de adquisición:
tiene margen para comprar mucho más crecimiento del que está comprando.

## 8. Volcado a `analysis/outputs/cac_ltv.json`

In [9]:
# La cifra sale del cálculo, no del teclado: escrita a mano se queda vieja.
_intensity_es = f"{marketing_spend / total_revenue * 100:.2f}".replace(".", ",")

payload = {
    "meta": {
        "page": "07_cac_ltv",
        "title": "CAC × LTV por canal",
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "inputs": {
            "attribution.json": attribution["meta"]["generated_at"],
            "cohorts_rfm.json": cohorts["meta"]["generated_at"],
        },
        "cac_measure": "cac_cargado_markov",
        "cac_measure_note": ("CAC cargado con el gasto huérfano: el 24,5% del gasto no resuelve a "
                             "ningún cliente y descartarlo abarata a unos canales más que a otros."),
        "ltv_measure": "ltv_proyectado",
        "ltv_measure_note": ("LTV proyectado con la curva de retención del canal, no el observado "
                             "sobre suscripciones maduras, que sólo promedia a los supervivientes "
                             "y premia al canal que peor retiene."),
        "attribution_cutoff": attribution["meta"]["censoring_cutoff"],
        "ltv_horizon_months": cohorts["meta"]["ltv_horizon_months"],
        "healthy_ratio": HEALTHY_RATIO,
        "payback_target_months": PAYBACK_TARGET_MONTHS,
    },
    "crossover": [
        {"channel": c, "label": CHANNEL_LABEL[c], "color": CHANNEL_COLOR[c],
         "cac_loaded_eur": float(priced.cac_cargado[c]),
         "cac_attributed_eur": float(priced.cac_reparto[c]),
         "ltv_projected_eur": float(priced.ltv_proyectado[c]),
         "ltv_observed_mature_eur": float(priced.ltv_observado_maduras[c]),
         "ratio": float(priced.ratio[c]),
         "payback_months": float(priced.payback_meses[c]),
         "margin_per_signup_eur": float(priced.margen_beneficio_eur[c]),
         "arpu_month_eur": float(priced.arpu_mes[c]),
         "expected_months": float(priced.meses_esperados[c]),
         "retention_12m_pct": float(priced.retencion_12m[c]),
         "conversions": float(priced.conversiones[c]),
         "verdict": str(recommendation.veredicto[c])}
        for c in priced.index],
    "unpriced_channels": [
        {"channel": c, "label": CHANNEL_LABEL[c],
         "ltv_projected_eur": float(unpriced.ltv_proyectado[c]),
         "retention_12m_pct": float(unpriced.retencion_12m[c]),
         "subscriptions": int(unpriced.suscripciones[c]),
         "reason": ("Sin coste de medios en el dataset: su coste real (contenido, SEO, marca) no "
                    "está modelado." if c == "organic" else
                    "No es un canal sino el cajón de las altas sin touchpoint resuelto.")}
        for c in unpriced.index],
    "robustness": {
        "note": ("El ranking depende de dos elecciones metodológicas: cargar o no el gasto "
                 "huérfano en el CAC, y usar LTV proyectado u observado sobre maduras."),
        "ratios": [
            {"channel": c, "label": CHANNEL_LABEL[c],
             **{k: float(v[c]) for k, v in combos.items()}}
            for c in ratios.index],
        "ranks": [
            {"channel": c, "label": CHANNEL_LABEL[c],
             **{k: int(ranks.loc[c, k]) for k in ranks.columns}}
            for c in ranks.index],
        "channels_changing_rank": int(len(movers)),
        "max_ratio_variation_pct": float((ratios.max(axis=1) / ratios.min(axis=1) - 1).max() * 100),
        "verdict": ("referral es el peor y el código de influencer el tercero en las cuatro "
                    "combinaciones. Paid social y podcast se intercambian el primer puesto según "
                    "el LTV que se use, y el sesgo de supervivencia del observado le da la "
                    "victoria justo al canal que peor retiene."),
    },
    "plausibility": {
        "note": ("Los ratios de 12x a 37x están muy por encima de lo razonable. Antes de "
                 "celebrarlo hay que buscar el coste que falta."),
        "marketing_spend_eur": float(marketing_spend),
        "total_revenue_eur": float(total_revenue),
        "marketing_intensity_pct": float(marketing_spend / total_revenue * 100),
        "typical_intensity_pct": "10-30",
        "no_cogs": True,
        "no_cogs_note": ("El dataset no tiene coste de producto, así que el LTV es ingreso y no "
                         "margen de contribución."),
        "ratio_by_margin": [
            {"margin_pct": float(m * 100),
             **{c: float(priced.ltv_proyectado[c] * m / priced.cac_cargado[c]) for c in priced.index}}
            for m in MARGIN_SCENARIOS],
        "breakeven_margin_pct": {c: float(v) for c, v in breakeven.items()},
        "verdict": ("Con un margen del 30% los ratios caen a 3,5x-11,2x, ya creíbles. Paid social "
                    "aguanta el umbral de 3x hasta con un 8% de margen; referral necesita un "
                    "25,4%, y es el único canal donde un supuesto plausible cambia la decisión."),
    },
    "blended": {
        "cac_loaded_eur": float(blended_cac),
        "ltv_weighted_eur": float(blended_ltv),
        "ratio": float(blended_ltv / blended_cac),
        "slowest_payback_months": float(recommendation.payback_meses.max()),
        "slowest_payback_channel": str(recommendation.payback_meses.idxmax()),
    },
    "insights": [
        ("El canal más barato de captar no es el mejor: el podcast cuesta 13,52 € por suscriptor "
         "frente a los 14,44 € de paid social, y aun así paid social vale más (ratio 37,3x frente "
         "a 33,7x) porque retiene diez puntos mejor a 12 meses."),
        ("La diferencia de LTV entre canales es casi toda retención: el ARPU mensual varía un 4,8% "
         "entre canales y los meses de vida esperados un 28%. El canal no cambia lo que el cliente "
         "paga, cambia cuánto tiempo se queda."),
        ("El referido cuesta tres veces más por alta que cualquier otro canal (40,57 €) y tiene el "
         "peor ratio (11,8x). Es el único donde el coste de captación es un problema real."),
        ("El ranking aguanta parcialmente las dos elecciones metodológicas: referral es el peor y "
         "el influencer el tercero en las cuatro combinaciones, pero paid social y podcast se "
         "intercambian el primer puesto según el LTV que se use."),
        ("Usar el LTV observado sobre suscripciones maduras no mete ruido: le da la victoria al "
         "canal que peor retiene, porque sólo promedia a los que sobrevivieron. El ratio de un "
         "mismo canal varía hasta un 66% según la combinación elegida."),
        ("Los ratios de 12x a 37x son demasiado buenos para ser una conclusión: el dataset no "
         "tiene coste de producto (el LTV es ingreso, no margen) y el gasto de marketing es el "
         f"{_intensity_es}% del ingreso cuando lo habitual en "
         "suscripción es un 10-30%."),
        ("Con un margen de contribución del 30% los ratios caen a 3,5x-11,2x, ya creíbles. Paid "
         "social seguiría sobre el umbral de 3x con un margen del 8%; referral necesita un 25,4%."),
        ("El payback más lento del catálogo es de 1,6 meses frente a un objetivo habitual de 12. "
         "Este negocio no tiene un problema de coste de adquisición: tiene margen para comprar "
         "más crecimiento del que compra."),
        ("El orgánico capta 914 suscripciones con 443 € de LTV y no aparece en ninguna tabla de "
         "eficiencia porque su coste no está en el dataset. Incluirlo es contabilidad analítica, "
         "no atribución."),
        ("El 29% de las altas entra por direct_unknown, sin origen resoluble. Su LTV (484 €) dice "
         "que no son peores clientes, pero están fuera del alcance de cualquier modelo de "
         "atribución."),
    ],
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as handle:
    json.dump(payload, handle, ensure_ascii=False, indent=2)
print(f"Guardado {OUTPUT_PATH.relative_to(PROJECT_ROOT)} "
      f"({OUTPUT_PATH.stat().st_size / 1024:.0f} KB)")
print("Claves de primer nivel:", list(payload))

Guardado analysis\outputs\cac_ltv.json (11 KB)
Claves de primer nivel: ['meta', 'crossover', 'unpriced_channels', 'robustness', 'plausibility', 'blended', 'insights']


In [10]:
with open(OUTPUT_PATH, encoding="utf-8") as handle:
    reloaded = json.load(handle)

checks = {
    "cuatro canales con precio": len(reloaded["crossover"]) == 4,
    "dos canales sin precio": len(reloaded["unpriced_channels"]) == 2,
    "todos los ratios positivos": all(r["ratio"] > 0 for r in reloaded["crossover"]),
    "el ratio cuadra con LTV/CAC": all(
        abs(r["ratio"] - r["ltv_projected_eur"] / r["cac_loaded_eur"]) < 1e-6
        for r in reloaded["crossover"]),
    "el payback cuadra con CAC/ARPU": all(
        abs(r["payback_months"] - r["cac_loaded_eur"] / r["arpu_month_eur"]) < 1e-6
        for r in reloaded["crossover"]),
    "el LTV cuadra con ARPU x meses": all(
        abs(r["ltv_projected_eur"] - r["arpu_month_eur"] * r["expected_months"]) < 0.01
        for r in reloaded["crossover"]),
    "el CAC cargado supera al del reparto": all(
        r["cac_loaded_eur"] >= r["cac_attributed_eur"] for r in reloaded["crossover"]),
    "el más barato no es el de mejor ratio": (
        min(reloaded["crossover"], key=lambda r: r["cac_loaded_eur"])["channel"]
        != max(reloaded["crossover"], key=lambda r: r["ratio"])["channel"]),
    "cuatro combinaciones de robustez": all(
        len(r) == 6 for r in reloaded["robustness"]["ratios"]),
    "hay canales que cambian de puesto": reloaded["robustness"]["channels_changing_rank"] > 0,
    "los rankings son permutaciones válidas": all(
        sorted(r[k] for r in reloaded["robustness"]["ranks"]) == [1, 2, 3, 4]
        for k in ("cargado × proyectado", "reparto × proyectado",
                  "cargado × obs. maduras", "reparto × obs. maduras")),
    "sensibilidad al margen": len(reloaded["plausibility"]["ratio_by_margin"]) == len(MARGIN_SCENARIOS),
    "el margen de equilibrio da el umbral": all(
        abs(reloaded["plausibility"]["breakeven_margin_pct"][r["channel"]] / 100
            * r["ltv_projected_eur"] / r["cac_loaded_eur"] - HEALTHY_RATIO) < 1e-6
        for r in reloaded["crossover"]),
    "intensidad de marketing por debajo del 3%": (
        reloaded["plausibility"]["marketing_intensity_pct"] < 3),
    "el ratio agregado cuadra": abs(
        reloaded["blended"]["ratio"]
        - reloaded["blended"]["ltv_weighted_eur"] / reloaded["blended"]["cac_loaded_eur"]) < 1e-6,
    "payback muy por debajo del objetivo": (
        reloaded["blended"]["slowest_payback_months"] < PAYBACK_TARGET_MONTHS),
}
for label, ok in checks.items():
    print(f"  {'OK ' if ok else 'FALLO'} {label}")
assert all(checks.values()), "El JSON de salida no tiene la forma esperada."
print()
print("JSON verificado.")

  OK  cuatro canales con precio
  OK  dos canales sin precio
  OK  todos los ratios positivos
  OK  el ratio cuadra con LTV/CAC
  OK  el payback cuadra con CAC/ARPU
  OK  el LTV cuadra con ARPU x meses
  OK  el CAC cargado supera al del reparto
  OK  el más barato no es el de mejor ratio
  OK  cuatro combinaciones de robustez
  OK  hay canales que cambian de puesto
  OK  los rankings son permutaciones válidas
  OK  sensibilidad al margen
  OK  el margen de equilibrio da el umbral
  OK  intensidad de marketing por debajo del 3%
  OK  el ratio agregado cuadra
  OK  payback muy por debajo del objetivo

JSON verificado.


## Conclusiones

1. **El canal más barato no es el mejor, y ésa era la tesis.** El podcast capta por 13,52 € y paid
   social por 14,44 €, pero paid social vale más (37,3x contra 33,7x) porque retiene diez puntos
   mejor a los 12 meses. Ordenar canales por CAC —que es lo que hace cualquier cuadro de mando de
   marketing— recomienda lo contrario que ordenarlos por valor.
2. **Y lo que separa a los canales es la retención, no la tarifa.** El ARPU varía un 4,8% entre
   canales y los meses de vida un 28%: el 85% de la diferencia de LTV viene de cuánto dura el
   cliente. La palanca no es el precio, que es común, sino la calidad de lo que trae cada canal.
3. **Las dos decisiones metodológicas de las páginas anteriores deciden el podio.** Usar el LTV
   observado sobre suscripciones maduras en vez del proyectado no añade ruido: **le da la victoria
   al canal que peor retiene**, porque sólo promedia supervivientes. El ratio de un mismo canal se
   mueve hasta un 66% según qué combinación se elija, y las cuatro tablas tienen el mismo aspecto
   de rigor.
4. **Lo que sí aguanta las cuatro combinaciones** es que `referral` es el peor canal —tres veces más
   caro que el resto— y que el código de influencer es tercero por su mala retención. Ésas son las
   dos conclusiones que se pueden defender sin asteriscos.
5. **Un ratio demasiado bueno es un hallazgo, no un logro.** De 12x a 37x está fuera de rango, y la
   explicación está en dos costes ausentes: el margen de producto, que el dataset no modela, y una
   intensidad de marketing del 1,84% del ingreso cuando lo normal es un 10-30%. Con un margen del
   30% los ratios vuelven a territorio creíble (3,5x-11,2x) sin cambiar el orden de los canales.
6. **La recomendación de negocio es incómoda pero es la que dicen los números**: con un payback
   máximo de 1,6 meses frente a un objetivo de 12, el cuello de botella de este negocio no es el
   coste de captación sino el volumen. Lo que hay que revisar no es el CAC de los canales buenos,
   es por qué se está comprando tan poco crecimiento.